In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import cv2
import numpy as np
from torchvision import transforms

# Класс для загрузки данных
class HomographyDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx, 0]
        print(f"Загрузка изображения: {image_path}")
        image = cv2.imread(image_path)
        if image is None:
            print(f"Ошибка: Не удалось загрузить изображение {image_path}.")
            return None
        print(f"Изображение {image_path} успешно загружено.")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Преобразуем в RGB
        if self.transform:
            image = self.transform(image)

        # Параметры гомографии
        homography_params = self.data.iloc[idx, 1:].values.astype(np.float32)
        homography_params = torch.tensor(homography_params, dtype=torch.float32)

        return image, homography_params

# Преобразования для изображений
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),  # Изменяем размер изображения
    transforms.ToTensor(),  # Преобразуем в тензор
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Нормализация
])

# Загрузка данных
dataset = HomographyDataset(csv_file="homography_data.csv", transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [9]:
import torch.nn as nn

class HomographyNet(nn.Module):
    def __init__(self):
        super(HomographyNet, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 32 * 32, 512),
            nn.ReLU(),
            nn.Linear(512, 8),  # 8 параметров гомографии
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)  # Разворачиваем в вектор
        x = self.fc(x)
        return x

# Создаем модель
model = HomographyNet()
print(model)

HomographyNet(
  (cnn): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=131072, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=8, bias=True)
  )
)


In [10]:
import torch.nn as nn

class HomographyNet(nn.Module):
    def __init__(self):
        super(HomographyNet, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 32 * 32, 512),
            nn.ReLU(),
            nn.Linear(512, 8),  # 8 параметров гомографии
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)  # Разворачиваем в вектор
        x = self.fc(x)
        return x

# Создаем модель
model = HomographyNet()
print(model)

HomographyNet(
  (cnn): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=131072, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=8, bias=True)
  )
)


In [12]:
import torch.optim as optim

# Функция потерь и оптимизатор
criterion = nn.MSELoss()  # Среднеквадратичная ошибка
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Обучение
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        # Обнуляем градиенты
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass и оптимизация
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

Загрузка изображения: peper/input_images\1frame_0064.jpg
Изображение peper/input_images\1frame_0064.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0106.jpg
Изображение peper/input_images\1frame_0106.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0011.jpg
Изображение peper/input_images\1frame_0011.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0044.jpg
Изображение peper/input_images\1frame_0044.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0015.jpg
Изображение peper/input_images\1frame_0015.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0001.jpg
Изображение peper/input_images\1frame_0001.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0098.jpg
Изображение peper/input_images\1frame_0098.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0025.jpg
Изображение peper/input_images\1frame_0025.jpg успешно загружено.
Загрузка изображ

In [13]:
model.eval()
test_loss = 0.0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

print(f"Test Loss: {test_loss/len(test_loader):.4f}")

Загрузка изображения: peper/input_images\1frame_0105.jpg
Изображение peper/input_images\1frame_0105.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0033.jpg
Изображение peper/input_images\1frame_0033.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0036.jpg
Изображение peper/input_images\1frame_0036.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0052.jpg
Изображение peper/input_images\1frame_0052.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0068.jpg
Изображение peper/input_images\1frame_0068.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0042.jpg
Изображение peper/input_images\1frame_0042.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0078.jpg
Изображение peper/input_images\1frame_0078.jpg успешно загружено.
Загрузка изображения: peper/input_images\1frame_0072.jpg
Изображение peper/input_images\1frame_0072.jpg успешно загружено.
Загрузка изображ